# Task 01 — Data Ingestion
**Agent:** `antigravity`
**Date:** `2026-03-16`
**Dataset:** `data/raw/AB_NYC_2019.csv`

**Objective:** Load dataset, check schema, assess data quality, justify cleaning decisions, and produce a clean 
`ingested.csv`.

In [1]:
# ── Imports & seed ────────────────────────────────────────────────
import random
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RAW_DIR    = Path('../../../data/raw')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Dataset
Understand what we are working with.

In [2]:
data_file = RAW_DIR / 'AB_NYC_2019.csv'
df = pd.read_csv(data_file)
print(f'Loaded: {data_file}')
print(f'Shape : {df.shape}')
df.head()

Loaded: ..\..\..\data\raw\AB_NYC_2019.csv
Shape : (48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


## 2. Schema Validation
What are the columns? What are their types?
Outputting to `schema_log.md` and `schema_log.txt`.

In [3]:
schema_lines = ['# Schema Summary\n']
schema_lines.append(f'**Row count:** {len(df)}\n')
schema_lines.append(f'**Column count:** {len(df.columns)}\n\n')
schema_lines.append('| Column | dtype |\n|--------|-------|')

for col, dtype in df.dtypes.items():
    schema_lines.append(f'| {col} | {dtype} |')

schema_md = '\n'.join(schema_lines)
(OUTPUT_DIR / 'schema_log.md').write_text(schema_md, encoding='utf-8')
(OUTPUT_DIR / 'schema_log.txt').write_text(schema_md, encoding='utf-8')
print('Saved schema_log.md and schema_log.txt\n')
print(schema_md)

Saved schema_log.md and schema_log.txt

# Schema Summary

**Row count:** 48895

**Column count:** 16


| Column | dtype |
|--------|-------|
| id | int64 |
| name | object |
| host_id | int64 |
| host_name | object |
| neighbourhood_group | object |
| neighbourhood | object |
| latitude | float64 |
| longitude | float64 |
| room_type | object |
| price | int64 |
| minimum_nights | int64 |
| number_of_reviews | int64 |
| last_review | object |
| reviews_per_month | float64 |
| calculated_host_listings_count | int64 |
| availability_365 | int64 |


## 3. Data Quality and Missingness
Identify missing values, produce missingness report, flag outliers.

In [4]:
# Missingness
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing_counts, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df.missing_count > 0].sort_values('missing_pct', ascending=False)

missing_df.to_csv(OUTPUT_DIR / 'missingness_report.csv')

report_lines = ['# Missingness Report\n']
if missing_df.empty:
    report_lines.append('No missing values found.')
else:
    report_lines.append(missing_df.to_markdown())
    report_lines.append('\n\n## Handling Strategy')
    report_lines.append('| Column | Strategy | Justification |')
    report_lines.append('|--------|----------|---------------|')
    report_lines.append('| `last_review` | Fill with `"No review"` | Null means no review left, not an unknown date. Sentinel string preserves this meaning without fabricating data. |')
    report_lines.append('| `reviews_per_month` | Fill with `0.0` | Structurally linked to `last_review`; zero reviews means a rate of 0. |')
    report_lines.append('| `name` | Fill with `"Unknown"` | ~0.02% missing. Sentinel keeps all rows; not heavily used for modelling. |')
    report_lines.append('| `host_name` | Fill with `"Unknown"` | ~0.01% missing. Sentinel keeps all rows. |')

(OUTPUT_DIR / 'missingness_report.md').write_text('\n'.join(report_lines), encoding='utf-8')
print('Saved missingness_report.csv and missingness_report.md\n')
print(missing_df)
print('\n')

# Outliers and Invalid Values
outlier_flags = []
# Minimum nights > 365
extreme_nights = df[df['minimum_nights'] > 365]
if not extreme_nights.empty:
    outlier_flags.append(f'minimum_nights: {len(extreme_nights)} rows > 365 days')

# Price <= 0 (Free listings or error)
zero_price = df[df['price'] <= 0]
if not zero_price.empty:
    outlier_flags.append(f'price: {len(zero_price)} rows with 0 or negative price (non-sensical)')

# High Price
high_price = df[df['price'] > 5000]
if not high_price.empty:
    outlier_flags.append(f'price: {len(high_price)} rows > $5000 per night')

(OUTPUT_DIR / 'outlier_flags.txt').write_text('\n'.join(outlier_flags), encoding='utf-8')
print('Saved outlier_flags.txt:')
for flag in outlier_flags:
    print('  -', flag)

Saved missingness_report.csv and missingness_report.md

                   missing_count  missing_pct
reviews_per_month          10052        20.56
last_review                10052        20.56
host_name                     21         0.04
name                          16         0.03


Saved outlier_flags.txt:
  - minimum_nights: 14 rows > 365 days
  - price: 11 rows with 0 or negative price (non-sensical)
  - price: 20 rows > $5000 per night


## 4. Make and Justify Cleaning Decisions

**Decisions:**
1. **Missing values:**
    - `name` & `host_name`: Fill with string 'Unknown'. These aren't heavily useful for modelling, but dropping them loses rows unnecessarily.
    - `last_review`: Fill with 'No review'. Null indicates an Airbnb with 0 reviews.
    - `reviews_per_month`: Fill with 0.0, corresponding to 0 reviews.
2. **Suspicious / Unusable Columns:**
    - `id` and `host_id`: Keep them for traceability. We don't drop columns arbitrarily yet.
3. **Outliers:**
    - `price`: Values <= 0 or >$5000 are likely errors or extreme outliers. However, **Constraint: Do NOT apply any transformation that uses information derived from price.** Therefore, we will **NOT** filter or remove price=0 rows or extreme price rows in this step to comply.
    - `minimum_nights` > 365: Some have ~10,000 days. To avoid introducing bias before EDA, and because missingness handling doesn't require filtering, we leave them be.

**Final Actions Applied:**
- Fill the 4 columns with missing values safely (No leakage).
- Do not drop any rows, preserving the full index to strictly follow 'do not transform target' and avoid over-filtering before EDA.

In [5]:
df['last_review'] = df['last_review'].fillna('No review')
df['reviews_per_month'] = df['reviews_per_month'].fillna(0.0)
df['name'] = df['name'].fillna('Unknown')
df['host_name'] = df['host_name'].fillna('Unknown')

print(f'Shape after handling: {df.shape}')
assert df.isnull().sum().sum() == 0, 'Still missing values'

Shape after handling: (48895, 16)


## 5. Produce Clean Dataset

In [6]:
out_path = OUTPUT_DIR / 'ingested.csv'
df.to_csv(out_path, index=False)
print(f'Saved {out_path} ({len(df)} rows)')

Saved outputs\ingested.csv (48895 rows)
